In [14]:
import lancedb
from ranx import Qrels, Run, evaluate, compare


In [2]:
db = lancedb.connect("../data/lancedb_mldr")


In [3]:
db.list_tables()


ListTablesResponse(context=None, tables=['train_reranking_passages', 'train_reranking_qrels', 'train_reranking_queries'], page_token=None)

In [56]:
tb_passages = db.open_table('train_reranking_passages')
tb_qrels = db.open_table('train_reranking_qrels')
tb_queries = db.open_table('train_reranking_queries')


In [5]:
tb_passages.count_rows()

4694

In [8]:
qs['query'][0]

'Quais são as principais fontes de divisas de Fiji?'

In [52]:
qrels = tb_qrels.search().where("query_id = 'q-pt-1385'").to_pandas()
qrels.sort_values(by="query_id", inplace=True)
qrels


,query_id,passage_id,score
0,q-pt-1385,doc-pt-9259,1


In [53]:
def query_pandas_dict(df, query_col, doc_col, score_col):
    qid_atual = None
    q_docs = {}
    qrels_dict = {}
    for _, q in df.iterrows():
        if qid_atual is None:
            qid_atual = q[query_col]
            q_docs = {}

        if qid_atual != q[query_col]:
            qrels_dict[qid_atual] = q_docs
            qid_atual = q[query_col]
            q_docs = {}
        q_docs[q[doc_col]] = q[score_col]
    qrels_dict[qid_atual] = q_docs
    return qrels_dict

In [54]:

qrels_dict = query_pandas_dict(qrels, "query_id", "passage_id", "score")
qrels_dataset = Qrels(qrels_dict)
print(len(qrels_dict))

1


In [60]:
MAX_RESULTS = 10
results_dict = {}
for k, v in qrels_dict.items():
    q = tb_queries.search().where(f"query_id = '{k}'").limit(1).to_pandas()
    s = tb_passages.search(query=q["query"].iloc[0], query_type="fts").select(["passage_id", "passage"]).limit(MAX_RESULTS).to_pandas()
    s["query_id"] = k
    qrels_dict = query_pandas_dict(s, "query_id", "passage_id", "_score")
    results_dict[k] = qrels_dict[k]

results_dict

[2026-11-08T02:32:46Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_score`.  Currently the `_score` column will be included.  In the future it will not.  Call `disable_scoring_autoprojection` to adopt the future behavior and avoid this warning


{'q-pt-1385': {'doc-pt-9259': 18.85467529296875,
  'doc-pt-12725': 12.054374694824219,
  'doc-pt-3840': 9.487428665161133,
  'doc-pt-8392': 8.897403717041016,
  'doc-pt-8669': 8.884546279907227,
  'doc-pt-5475': 8.658050537109375,
  'doc-pt-5950': 8.134687423706055,
  'doc-pt-7254': 8.098160743713379,
  'doc-pt-15498': 8.039491653442383,
  'doc-pt-8859': 7.884558200836182}}

In [43]:
s['query_id'] = 'q-pt-1385'
qrels_dict = query_pandas_dict(s, "query_id", "passage_id", "_score")
bm25_dataset = Run(qrels_dict, name="bm25")
print(len(bm25_dataset))

1


In [46]:
# -------------------------------------------------------------
# Avaliação de um único sistema
# -------------------------------------------------------------
metrics = ["ndcg@10", "map@10", "mrr@10", "recall@10", "hit_rate@10"]
scores = evaluate(qrels_dataset, bm25_dataset, metrics)

print("Métricas Sistema Denso:")
for m, v in scores.items():
    print(f"  {m:<12}: {v:.4f}")


/home/gvanerven/code/doutorado/.venv/lib/python3.12/site-packages/ranx/metrics/ndcg.py:72: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _ndcg(qrels[i], run[i], k, rel_lvl, jarvelin)


Métricas Sistema Denso:
  ndcg@10     : 1.0000
  map@10      : 1.0000
  mrr@10      : 1.0000
  recall@10   : 1.0000
  hit_rate@10 : 1.0000


In [ ]:
# -------------------------------------------------------------
# Comparação entre modelos com teste estatístico
# -------------------------------------------------------------
report = compare(
    qrels=qrels_dataset,
    runs=[run_bm25, run_dense],
    metrics=["ndcg@10", "mrr@10", "recall@10"],
    stat_test="student_t"  # Realiza teste t pareado
)
print("\nRelatório Comparativo:")
print(report)